# Apex Qwen Coder: model demo

Load the **verified final augmented-continuation adapter**, recorded at
222/269 (82.53%) test and 184/192 (95.83%) validation. No training runs here.

The interface has **MCQ mode** and experimental **QA mode**. MCQ mode retains
the evaluated prompt/parser. QA mode asks for a direct explanation with a
separate system instruction; the MCQ accuracy does not apply to it.

This notebook reuses `organized/space/` app code and adapter. A dedicated CUDA
GPU and enough memory for the 4-bit 7B model are required for this notebook.
For ZeroGPU, deploy the same app folder to the Hugging Face Space instead.
Python3.12 is recommended. The original protected assessment notebook is unchanged.


In [ ]:
from pathlib import Path
import sys

# Find the organized folder whether Jupyter starts here, in organized/, or at the project root.
candidates = []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / 'organized'])
ORGANIZED = next((p.resolve() for p in candidates
                  if (p / 'space/app.py').is_file() and (p / 'data/manifest.json').is_file()), None)
if ORGANIZED is None:
    raise FileNotFoundError('Open this notebook from the organized project folder.')
SPACE = ORGANIZED / 'space'
print('Python:', sys.version.split()[0])
print('App folder:', SPACE)


## Install dependencies

Run this cell once in a fresh GPU environment. If this kernel already imported
Torch/Transformers/Gradio, restart it after installation, rerun the path cell,
and continue with adapter verification. The first model load downloads the
pinned public Qwen base; the final adapter is already in `space/adapter/`.


In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(SPACE / 'requirements.txt')], check=True)


## Verify the final adapter and runtime

The app refuses missing, changed, or older adapter weights. Nothing is uploaded by this notebook.


In [ ]:
import os
os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('GRADIO_ANALYTICS_ENABLED', 'false')
sys.path.insert(0, str(SPACE))
from inference_contract import verify_adapter
manifest = verify_adapter(SPACE)
print('Run:', manifest['run_id'])
print('Adapter SHA-256:', manifest['sha256']['adapter_model.safetensors'])
print('Recorded MCQ test accuracy:', f"{manifest['test']['accuracy']:.2%}")
print('Recorded MCQ validation accuracy:', f"{manifest['validation']['accuracy']:.2%}")

import spaces  # Import before Torch for compatibility with the deployed app.
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Use a CUDA GPU runtime for this local notebook, or deploy space/ on ZeroGPU.')
print('GPU:', torch.cuda.get_device_name(0))


## Open MCQ and QA modes

Model loading can take time on the first run. MCQ mode needs a question and four
distinct choices. QA mode needs only a question; optional project code can help.
The notebook uses the final adapter in both modes. The interface is local/inline;
`share=False` does not create a public Gradio tunnel.


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('apex_qwen_demo', SPACE / 'app.py')
apex = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = apex
spec.loader.exec_module(apex)
if not apex.runtime.ready:
    raise RuntimeError(apex.runtime.problem)
apex.demo.launch(inline=True, share=False, show_error=False)


## Finish

Use `apex.demo.close()` to close the interface. The notebook does not change
weights, regenerate training data or publish the Space. For recorded experiment
results, read `organized/full-experiment-workflow.md`.
